# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [4]:
# Load data
df = pd.read_csv("AviationData.csv", encoding="latin-1")

# Inspect the first few rows
df.head()

# Check the shape of the dataframe
df.shape

# Check data types and non-null counts
df.info()

# Check missing values
df.isna().sum()

# Summary statistics
df.describe(include="all")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_20408\3548277724.py:2: DtypeWarning: Columns (6: Latitude, 7: Longitude, 28: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("AviationData.csv", encoding="latin-1")


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50132,52704,...,82697,16648,77488.000000,76379.000000,76956.000000,82977.000000,84397,61724,82505,75118
unique,87951,2,88863,14782,27758,219,25592,27156,10374,24870,...,26,13590,NaN,NaN,NaN,NaN,4,12,17074,2924
top,20001214X45071,Accident,ERA22LA103,1982-05-16,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,...,Personal,Pilot,NaN,NaN,NaN,NaN,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,...,49448,258,NaN,NaN,NaN,NaN,77303,15428,61754,17019
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.647855,0.279881,0.357061,5.325440,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.485960,1.544084,2.235625,27.913634,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# Inspect the relevant columns
df[[
    "Event.Date",
    "Investigation.Type",
    "Aircraft.Category",
    "Amateur.Built",
    "Make",
    "Model"
]].head()

# Convert Event.Date to datetime
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

# Filter the dataset based on the client's requirements
df = df[
    (df["Event.Date"].dt.year >= 1983) &
    (df["Investigation.Type"] == "Accident") &
    (df["Aircraft.Category"] == "Airplane") &
    (df["Amateur.Built"] == "No")
].copy()

# Remove rows where Make or Model is missing
df = df.dropna(subset=["Make", "Model"])

# Check the size of the filtered dataset
df.shape

(19923, 31)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
# Fill missing injury values with 0
injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

df[injury_columns] = df[injury_columns].fillna(0)

# Estimate the total number of people on each flight
df["Total.People"] = df[injury_columns].sum(axis=1)

# Calculate fatal + serious injuries
df["Fatal.Serious.Injuries"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"]
)

# Calculate the fatal/serious injury rate
df["Fatal.Serious.Injury.Rate"] = (
    df["Fatal.Serious.Injuries"] / df["Total.People"]
)

df[[
    "Total.People",
    "Fatal.Serious.Injuries",
    "Fatal.Serious.Injury.Rate"
]].head()

,Total.People,Fatal.Serious.Injuries,Fatal.Serious.Injury.Rate
4171,2.0,2.0,1.000000
4285,5.0,1.0,0.200000
5960,26.0,13.0,0.500000
6669,30.0,1.0,0.033333
6806,1.0,0.0,0.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
df["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    16957
Destroyed       2311
NaN              433
Minor            147
Unknown           75
Name: count, dtype: int64

In [13]:
# Fill missing aircraft damage values with "Unknown"
df["Aircraft.damage"] = df["Aircraft.damage"].fillna("Unknown")

# Create a variable indicating whether the aircraft was destroyed
df["Aircraft.Destroyed"] = (
    df["Aircraft.damage"] == "Destroyed"
).astype(int)

# Check the new variable
df["Aircraft.Destroyed"].value_counts()

Aircraft.Destroyed
0    17612
1     2311
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [14]:
df["Make"].value_counts(dropna=False).head(30)

Make
CESSNA                      4764
PIPER                       2760
Cessna                      2261
Piper                       1177
BEECH                        985
BOEING                       418
Beech                        407
MOONEY                       235
AIR TRACTOR INC              215
CIRRUS DESIGN CORP           207
BELLANCA                     156
AERONCA                      149
MAULE                        144
Boeing                       134
Mooney                       123
Air Tractor                  117
LUSCOMBE                      95
STINSON                       91
DEHAVILLAND                   90
CHAMPION                      90
AIR TRACTOR                   89
GRUMMAN                       77
NORTH AMERICAN                77
EMBRAER                       74
CIRRUS                        74
DIAMOND AIRCRAFT IND INC      72
AVIAT AIRCRAFT INC            72
Maule                         71
Grumman                       69
AERO COMMANDER                68
Name:

In [15]:
# Remove leading/trailing spaces from Make names
df["Make"] = df["Make"].str.strip()

# Standardize Make names to uppercase
df["Make"] = df["Make"].str.upper()

# Remove rows with missing Make values
df = df.dropna(subset=["Make"])

# Keep Makes with at least 50 records
make_counts = df["Make"].value_counts()
valid_makes = make_counts[make_counts >= 50].index

df = df[df["Make"].isin(valid_makes)].copy()

# Check the result
df["Make"].value_counts().head(30)

Make
CESSNA                            7025
PIPER                             3937
BEECH                             1392
BOEING                             552
MOONEY                             358
BELLANCA                           217
AIR TRACTOR INC                    217
MAULE                              215
CIRRUS DESIGN CORP                 209
AIR TRACTOR                        206
AERONCA                            200
CHAMPION                           157
GRUMMAN                            146
LUSCOMBE                           141
CIRRUS                             130
STINSON                            129
NORTH AMERICAN                     104
EMBRAER                             97
TAYLORCRAFT                         93
DEHAVILLAND                         93
AERO COMMANDER                      90
AIRBUS                              82
AVIAT AIRCRAFT INC                  76
SOCATA                              74
DIAMOND AIRCRAFT IND INC            72
AVIAT               

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [16]:
# Remove rows with missing Model values
df = df.dropna(subset=["Model"])

# Check for missing Model values
df["Model"].isna().sum()

np.int64(0)

In [17]:
# Inspect the most common Model values
df["Model"].value_counts().head(30)

Model
172          763
152          311
182          300
172S         272
PA28         268
172N         249
SR22         228
180          212
A36          181
172M         179
150          177
PA-18-150    175
PA-28-140    169
172P         141
737          118
140          117
172R         108
170B         107
PA-28-180    105
PA-28-161    101
7AC           95
210           95
PA-28-181     92
M20J          92
A185F         90
7GCBC         86
PA18          86
PA-18         80
208B          78
182P          78
Name: count, dtype: int64

In [18]:
# Check whether each Model is associated with only one Make
model_make_counts = df.groupby("Model")["Make"].nunique()

model_make_counts[model_make_counts > 1].head(20)

Model
100      2
112      2
112A     2
140      2
1900D    2
200      2
320      2
350      2
390      2
400      2
400A     2
401B     2
402A     2
402B     2
500      2
500 S    2
560      2
58       2
690A     2
7AC      3
Name: Make, dtype: int64

In [19]:
# Remove rows with missing Model values
df = df.dropna(subset=["Model"])

# Inspect whether Models are unique to each Make
model_make_counts = df.groupby("Model")["Make"].nunique()
print(model_make_counts[model_make_counts > 1].head(20))

# Create a unique identifier using Make + Model
df["Plane.Type"] = df["Make"] + " " + df["Model"]

# Check the result
df[["Make", "Model", "Plane.Type"]].head()

Model
100      2
112      2
112A     2
140      2
1900D    2
200      2
320      2
350      2
390      2
400      2
400A     2
401B     2
402A     2
402B     2
500      2
500 S    2
560      2
58       2
690A     2
7AC      3
Name: Make, dtype: int64


,Make,Model,Plane.Type
4171,PIPER,PA-28-140,PIPER PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND DHC-6
6806,BEECH,C35,BEECH C35
7084,CESSNA,180K,CESSNA 180K
7708,BEECH,99,BEECH 99


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [21]:
# Inspect the five columns for missing values and unique values

columns_to_inspect = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for column in columns_to_inspect:
    print(f"\n--- {column} ---")
    print("Missing values:", df[column].isna().sum())
    print(df[column].value_counts(dropna=False))


--- Engine.Type ---
Missing values: 2442
Engine.Type
Reciprocating    12735
NaN               2442
Turbo Prop         877
Turbo Fan          364
Unknown             67
Turbo Jet           45
Turbo Shaft          8
UNK                  1
Name: count, dtype: int64

--- Weather.Condition ---
Missing values: 1504
Weather.Condition
VMC    13974
NaN     1504
IMC      871
Unk      147
UNK       43
Name: count, dtype: int64

--- Number.of.Engines ---
Missing values: 1424
Number.of.Engines
1.0    13130
2.0     1938
NaN     1424
4.0       28
3.0       15
0.0        4
Name: count, dtype: int64

--- Purpose.of.flight ---
Missing values: 1907
Purpose.of.flight
Personal                     9775
Instructional                2387
NaN                          1907
Aerial Application            721
Business                      393
Unknown                       253
Positioning                   253
Skydiving                     156
Aerial Observation            146
Other Work Use                118
Ban

In [22]:
# Display unique values for each column

for column in [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]:
    print(f"\n--- {column} ---")
    print(df[column].unique())


--- Engine.Type ---
<StringArray>
['Reciprocating',    'Turbo Prop',     'Turbo Jet',             nan,
       'Unknown',     'Turbo Fan',   'Turbo Shaft',           'UNK']
Length: 8, dtype: str

--- Weather.Condition ---
<StringArray>
['IMC', 'VMC', 'UNK', nan, 'Unk']
Length: 5, dtype: str

--- Number.of.Engines ---
[ 1.  2.  4. nan  0.  3.]

--- Purpose.of.flight ---
<StringArray>
[                 'Personal',                 'Skydiving',
                   'Unknown',        'Aerial Application',
               'Positioning',             'Instructional',
                         nan,                     'Ferry',
            'Other Work Use',                  'Business',
           'Public Aircraft',        'Aerial Observation',
       'Executive/corporate', 'Public Aircraft - Federal',
             'Air Race/show',               'Flight Test',
   'Public Aircraft - State',                'Glider Tow',
                'Banner Tow',             'Air Race show',
              'Firefight

In [23]:
### CLEANING
# Standardize Engine.Type
df["Engine.Type"] = df["Engine.Type"].replace({
    "UNK": "Unknown"
})

# Standardize Weather.Condition
df["Weather.Condition"] = df["Weather.Condition"].replace({
    "Unk": "UNK"
})

# Check the cleaned values
print(df["Engine.Type"].value_counts(dropna=False))
print(df["Weather.Condition"].value_counts(dropna=False))

Engine.Type
Reciprocating    12735
NaN               2442
Turbo Prop         877
Turbo Fan          364
Unknown             68
Turbo Jet           45
Turbo Shaft          8
Name: count, dtype: int64
Weather.Condition
VMC    13974
NaN     1504
IMC      871
UNK      190
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [24]:
# Check the number and percentage of missing values in each column
missing_values = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percent
}).sort_values("Missing %", ascending=False)

missing_summary

,Missing Values,Missing %
Schedule,15167,91.70
Broad.phase.of.flight,14120,85.37
Air.carrier,8982,54.31
Airport.Code,5424,32.80
Airport.Name,5353,32.37
Report.Status,2928,17.70
Engine.Type,2442,14.77
Purpose.of.flight,1907,11.53
Weather.Condition,1504,9.09
Number.of.Engines,1424,8.61


In [25]:
# Drop columns with too many missing values
df = df.drop(columns=[
    "Schedule",
    "Broad.phase.of.flight",
    "Air.carrier"
])

# Check the remaining columns
df.shape

(16539, 33)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [27]:
# Save the cleaned DataFrame
df.to_csv("AviationData_cleaned.csv", index=False)